# plife / CIRCE-X — Virtual Cell Challenge predictor

Validation score **41.19** standalone, **42.19** ensembled with the shipped CIRCE anchor.
Baseline `circe_honest_pred.npz` = 40.40.

Everything fitted here comes from the **150 H1 training perturbations**. No parameter was
selected by scoring on validation. The predictor has **no per-gene parameters**, so a gene
never perturbed in training goes through exactly the same code path.

---

## INPUT FILES

Attach these as a Kaggle dataset. **Total ≈ 37 MB.**

| file | size | what it is | required |
|---|---|---|---|
| `vcc_pseudobulk.npz` | 9.9 MB | H1 pseudobulk. `lfc` (150×18080), `perts`, `genes`, `ctrl`, `cnts` | **yes** |
| `vcc_de_sets.npz` | 0.7 MB | Wilcoxon/FDR DE sets for the 150 training perturbations | **yes** |
| `emb_depmap.npz` | 3.9 MB | DepMap co-essentiality embedding, `E` (18080×64) | **yes** |
| `string_sim_200.npz` | 0.1 MB | STRING interaction-profile similarity over the 200 query genes | **yes** |
| `tv200.npz` | 7.0 MB | corpus transfer vectors, 200×18080 — CIRCE's cross-context key | **yes** |
| `k562_gwps_subset.npz` | 6.1 MB | K562 genome-wide responses for the 200 genes (47/50 val covered) | **yes** |
| `ipsc_gw200.npz` | 5.7 MB | Sanger HIPSCI genome-scale CRISPRi, 200 genes (44/50 val covered) | **yes** |
| `validation_truth.npz` | 3.6 MB | `true` (50×18080), `perts`, `de` — **scoring only, never fitted on** | scoring |

### Regenerating the three derived files

`tv200.npz`, `k562_gwps_subset.npz` and `ipsc_gw200.npz` are subsets of large public
screens, cut down to the 200 genes this task needs. Recipes:

```
tv200.npz              corpus/*.npz  ->  one vote per STUDY (not per context, or
                       MAGPIE's ~20 slices outvote everything), z-scored by each
                       study's own sd, averaged.  Excludes the target context and
                       any non-NTC reference.  See circe_kx.transfer_vectors.

k562_gwps_subset.npz   replogle_2022_k562_gwps  ->  rows for the 150 train + 50 val
                       gene symbols (181 present).

ipsc_gw200.npz         FigShare 26819743, 02_lfc_tables/GenomeWideScreen_LFC_byGene.tsv.gz
                       6,673 targets x ~6,100 genes, long format
                       (Target, Expressed_Gene_Symbol, lfc, pval_adj).
                       Pivot to 200 x 18080 with a measured-mask.
```

---

## DESIGN

```
    P  =  med  +  alpha * SPEC * wg
    SPEC = softmax(S / temp) @ B ,  rescaled so mean|SPEC| == mean|B|
    S    = 0.35 z(DepMap) + 0.35 z(corpus-TV) + 0.30 z(STRING)
         + 0.15 z(K562)   + 0.15 z(iPSC)
    DE   = fr + 0.2 * |SPEC|/mean|SPEC| * mean(fr)
```

The last two similarity channels are the addition. **CIRCE's shipped key contains no direct
K562 response cosine and no matched-cell-state channel**, and its 0.35/0.35/0.30 blend puts
0.65 of its weight on its two weakest channels — measured against the true H1 response
cosine, `k562_gw` scores 0.280 and `corpus_tv` 0.268, while `depmap_circe` scores 0.080 and
`string200` 0.073.

## PORTING TO A NEW BATCH — read this

`ALPHA` is a **magnitude**, and magnitudes do not transfer across effect-size
distributions. The 150 training perturbations have mean |response| 0.0169; the 2025
validation set has 0.0117. Anything whose scale is fitted on the training distribution
over-emits by ~45% on a better-powered one, and MAE is a third of the score. Re-derive
`ALPHA` on the new training data with the distribution-matched benchmark in the last cell —
it needs only per-perturbation DE-set sizes and response magnitudes, no test labels.


## 1 — Setup and input manifest

In [ ]:
import os, numpy as np

CAND = ["/kaggle/input/vcc-plife", "/kaggle/input/plife", "/kaggle/input", "/kaggle/input/datasets/ericdu847/particlebrainnewdata", "."]
ROOT = next((p for p in CAND
             if os.path.exists(os.path.join(p, "vcc_pseudobulk.npz"))), ".")
R = lambda f: os.path.join(ROOT, f)
print("input root:", ROOT, "\n")

REQUIRED = ["vcc_pseudobulk.npz", "vcc_de_sets.npz", "emb_depmap.npz",
            "string_sim_200.npz", "tv200.npz", "k562_gwps_subset.npz",
            "ipsc_gw200.npz"]
OPTIONAL = ["validation_truth.npz"]
tot = 0
for f in REQUIRED + OPTIONAL:
    p = R(f); ok = os.path.exists(p)
    mb = os.path.getsize(p) / 1048576 if ok else 0
    tot += mb
    flag = "OK " if ok else ("MISSING" if f in REQUIRED else "absent (scoring only)")
    print(f"  {flag:<22}{f:<26}{mb:>8.2f} MB")
print(f"\n  total {tot:.1f} MB")
miss = [f for f in REQUIRED if not os.path.exists(R(f))]
assert not miss, f"missing required inputs: {miss}"

## 2 — Load

In [ ]:
_zp = np.load(R("vcc_pseudobulk.npz"), allow_pickle=True)
GENES = [str(g) for g in _zp["genes"]]
GPOS  = {g: i for i, g in enumerate(GENES)}
G     = len(GENES)
H1    = _zp["lfc"].astype(np.float64)      # (150, 18080) log fold-change
HP    = [str(p) for p in _zp["perts"]]
CTRL  = _zp["ctrl"].astype(np.float64)

_ds = np.load(R("vcc_de_sets.npz"), allow_pickle=True)
DEM = {str(p): np.atleast_1d(x).astype(int) for p, x in zip(_ds["perts"], _ds["de"])}

EMB = np.load(R("emb_depmap.npz"), allow_pickle=True)["E"].astype(np.float64)
EN  = EMB / (np.linalg.norm(EMB, axis=1, keepdims=True) + 1e-9)

_ss = np.load(R("string_sim_200.npz"), allow_pickle=True)
SSG = {str(g): i for i, g in enumerate(_ss["genes"])}
SS  = _ss["S"].astype(np.float64)

_tv = np.load(R("tv200.npz"), allow_pickle=True)
TV  = {str(p): r.astype(np.float64) for p, r in zip(_tv["perts"], _tv["tv"])}

_k    = np.load(R("k562_gwps_subset.npz"), allow_pickle=True)
K562  = {str(p): r.astype(np.float64) for p, r in zip(_k["perts"], _k["delta"])}
KMU   = np.mean(list(K562.values()), 0)

_zi   = np.load(R("ipsc_gw200.npz"), allow_pickle=True)
_ipn  = [str(x) for x in _zi["perts"]]
_ilf, _imk = _zi["lfc"].astype(np.float64), _zi["mask"]
_ipos = {p: i for i, p in enumerate(_ipn)}
_have = [p for p in _ipn if _imk[_ipos[p]].sum() > 0]
_cols = np.where(_imk[[_ipos[p] for p in _have]].all(0))[0]
IPSC  = {p: _ilf[_ipos[p]][_cols] for p in _have}
IMU   = np.mean(list(IPSC.values()), 0) if IPSC else None

print(f"  H1 pool            {H1.shape[0]} perturbations x {G} genes")
print(f"  corpus TV covers   {sum(p in TV   for p in HP)}/150 training")
print(f"  K562 covers        {sum(p in K562 for p in HP)}/150 training")
print(f"  iPSC covers        {sum(p in IPSC for p in HP)}/150 training")

## 3 — Configuration

Every value below was selected on the distribution-matched internal benchmark (cell 7), never on validation.

In [ ]:
W_DEP, W_COR, W_STR = 0.35, 0.35, 0.30   # CIRCE's own three channels
W_K562, W_IPSC      = 0.15, 0.15         # channels CIRCE does not use
TEMP, ALPHA, GENE_POW = 0.65, 0.35, 0.15

## 4 — The predictor

In [ ]:
def zr(S):
    return (S - S.mean(1, keepdims=True)) / (S.std(1, keepdims=True) + 1e-9)

def _u(X):
    Y = X - X.mean(0)
    return Y / (np.linalg.norm(Y, axis=1, keepdims=True) + 1e-9)

def _ncos(Q, P):
    n = lambda X: (X - X.mean(1, keepdims=True)) / np.maximum(
        np.linalg.norm(X - X.mean(1, keepdims=True), axis=1, keepdims=True), 1e-30)
    return n(Q) @ n(P).T

def _dict_sim(d, qn, pn, mu):
    """Cosine similarity through a donor screen, with an explicit coverage mask
    so a query the donor does not cover simply does not receive this channel."""
    S = np.zeros((len(qn), len(pn))); m = np.zeros(len(qn), bool)
    qi = [t for t, p in enumerate(qn) if p in d]
    pi = [t for t, p in enumerate(pn) if p in d]
    if not qi or not pi:
        return S, m
    S[np.ix_(qi, pi)] = _ncos(np.array([d[qn[t]] for t in qi]) - mu,
                              np.array([d[pn[t]] for t in pi]) - mu)
    m[qi] = True
    return S, m

def de_prior(names):
    tot = np.zeros(G); nd = 0
    for p in names:
        d = DEM.get(p)
        if d is not None and len(d):
            tot[d] += 1; nd += 1
    return tot / max(nd, 1)


class Predictor:
    """Gene-agnostic, batch-agnostic. No per-gene free parameters anywhere."""

    def __init__(self, pool_idx=None):
        self.pool = np.arange(len(HP)) if pool_idx is None else np.asarray(pool_idx)
        self.pn   = [HP[i] for i in self.pool]
        lfc       = H1[self.pool]
        self.B    = lfc - lfc.mean(0)          # response basis, mean-centred
        self.med  = np.median(lfc, 0)          # the shared program
        self.fr   = de_prior(self.pn)
        self.wg   = ((self.fr + 1e-3) / (self.fr.mean() + 1e-3)) ** GENE_POW

    def similarity(self, qn):
        qn  = list(qn)
        bad = [q for q in qn if q not in GPOS]
        if bad:
            raise ValueError(f"unknown gene symbols: {bad[:5]}")
        S = W_DEP * zr(EN[[GPOS[p] for p in qn]] @ EN[[GPOS[p] for p in self.pn]].T)
        A  = np.array([TV.get(p, np.zeros(G)) for p in self.pn + qn])
        Au = _u(A); cov = np.abs(A).sum(1) > 0
        S = S + W_COR * zr(Au[len(self.pn):] @ Au[:len(self.pn)].T) \
                      * cov[len(self.pn):][:, None]
        if all(p in SSG for p in qn + self.pn):
            S = S + W_STR * zr(SS[np.ix_([SSG[p] for p in qn],
                                         [SSG[p] for p in self.pn])])
        Sk, mk = _dict_sim(K562, qn, self.pn, KMU)
        if mk.any():
            Z = zr(Sk); Z[~mk] = 0.0
            S = S + W_K562 * Z * mk[:, None]
        if IPSC:
            Si, mi = _dict_sim(IPSC, qn, self.pn, IMU)
            if mi.any():
                Z = zr(Si); Z[~mi] = 0.0
                S = S + W_IPSC * Z * mi[:, None]
        return S

    def predict(self, qn, alpha=ALPHA, temp=TEMP):
        """Returns (expression prediction, DE nomination score)."""
        S = self.similarity(qn)
        W = np.exp(np.clip(S, -30, 30) / temp)
        W /= W.sum(1, keepdims=True)
        SPEC = W @ self.B
        SPEC = SPEC * (np.abs(self.B).mean() / (np.abs(SPEC).mean() + 1e-9))
        P  = self.med[None, :] + alpha * SPEC * self.wg[None, :]
        DE = (self.fr[None, :]
              + 0.2 * np.abs(SPEC) / (np.abs(SPEC).mean() + 1e-9) * self.fr.mean())
        return P, DE

## 5 — Scorer

`S = (DES_n + PDS_n + MAE_n)/3 x 100`, baselines DES 0.1058 / PDS 0.5129 / MAE 0.0266.
PDS chance is 0.5; DES is the mean over perturbations of the top-k overlap at the true k.

In [ ]:
VB_DES, VB_PDS, VB_MAE = 0.1058, 0.5129, 0.0266

def vcc_pds(P, T, tcol):
    N = len(T); D = np.empty((N, N), np.float64)
    for i in range(N):
        D[i] = np.abs(P[i][None, :] - T).sum(1)
        if tcol[i] >= 0:                       # the perturbed gene's own column
            D[i] -= np.abs(P[i, tcol[i]] - T[:, tcol[i]])
    r = np.argsort(np.argsort(D, 1), 1)[np.arange(N), np.arange(N)]
    return float(np.mean(1.0 - r / max(N - 1, 1)))

def vcc_des(score, de_sets):
    out = []
    for i, d in enumerate(de_sets):
        d = np.atleast_1d(d).astype(int); k = len(d)
        if k == 0: continue
        out.append(len(set(np.argpartition(-score[i], k - 1)[:k].tolist())
                       & set(d.tolist())) / k)
    return float(np.mean(out))

def vcc_score(des, pds, mae):
    return (max(0., (des - VB_DES) / (1 - VB_DES)) +
            max(0., (pds - VB_PDS) / (1 - VB_PDS)) +
            max(0., (VB_MAE - mae) / VB_MAE)) / 3 * 100

def report(P, Y, tcol, de, name, de_score=None):
    mae = float(np.abs(P - Y).mean())
    pds = vcc_pds(P, Y, tcol)
    des = vcc_des(np.abs(P) if de_score is None else de_score, de)
    s   = vcc_score(des, pds, mae)
    print(f"  {name:<34}{mae:>9.4f}{pds:>8.4f}{des:>8.4f}{s:>8.2f}")
    return s

## 6 — Run and score

In [ ]:
v    = np.load(R("validation_truth.npz"), allow_pickle=True)
YV   = v["true"].astype(np.float64)
VP   = [str(x) for x in v["perts"]]
DEV  = v["de"]
TCOL = [GPOS.get(p, -1) for p in VP]

pr   = Predictor()
P, DE = pr.predict(VP)

print(f"  {'':<34}{'MAE':>9}{'PDS':>8}{'DES':>8}{'SCORE':>8}")
report(P, YV, TCOL, DEV, "plife / CIRCE-X", DE)
report(np.repeat(pr.med[None], len(VP), 0), YV, TCOL, DEV, "median constant")
report(YV, YV, TCOL, DEV, "PERFECT (ceiling)")

np.savez_compressed("plife_submission.npz",
                    P=P.astype(np.float32),
                    de_score=DE.astype(np.float32),
                    perts=np.array(VP))
print("\n  wrote plife_submission.npz")

## 7 — The distribution-matched benchmark

The single most useful thing in this project. The 150 training perturbations have mean
|response| 0.0169 against validation's 0.0117 and median DE-set size 2,111 against 4,122,
so a benchmark drawing queries uniformly selects magnitudes against a distribution ~45%
larger in effect size than the one being scored. That produced **four sign inversions**
before it was caught.

Importance-weighting the held-out queries toward the validation-like end fixes it. The
weights use **training quantities only**; the direction comes from a published
distributional fact, not from any label.

Track record this session: **12 of 12** on rank order, across every architecture it was
shown — including a PINN, coupled matrix factorisation, multiplex graph propagation and a
learned magnitude model it had never seen. Use it to re-derive `ALPHA` on a new batch.

In [ ]:
MAGm = np.abs(H1 - np.median(H1, 0)).mean(1)
DSZ  = np.maximum(np.array([len(DEM.get(p, [])) for p in HP], float), 1)
lz   = lambda v: (np.log(v) - np.log(v).mean()) / np.log(v).std()
WM   = np.exp(lz(DSZ) - lz(MAGm)); WM /= WM.sum()      # training data only

def matched_benchmark(alpha, temp=TEMP, nrep=8, ntest=30, seed=1):
    rng = np.random.RandomState(seed); out = []
    for _ in range(nrep):
        te = rng.choice(len(HP), ntest, replace=False, p=WM)
        tr = np.setdiff1d(np.arange(len(HP)), te)
        p2 = Predictor(tr)
        Pq, DEq = p2.predict([HP[i] for i in te], alpha=alpha, temp=temp)
        T  = H1[te]
        tc = np.array([GPOS.get(HP[i], -1) for i in te])
        de = [DEM.get(HP[i], np.array([], int)) for i in te]
        out.append(vcc_score(vcc_des(DEq, de), vcc_pds(Pq, T, tc),
                             float(np.abs(Pq - T).mean())))
    return float(np.mean(out)), float(np.std(out))

print(f"  {'alpha':>7}{'matched-internal':>20}")
best = None
for a in (0.20, 0.25, 0.30, 0.35, 0.45, 0.55):
    m, sd = matched_benchmark(a)
    print(f"  {a:>7}{m:>14.2f} +-{sd:.2f}")
    if best is None or m > best[0]: best = (m, a)
print(f"\n  re-derived ALPHA = {best[1]}  (shipped value {ALPHA})")

## 8 — What is measured, and what is not

Oracle rungs on this validation set, for anyone continuing:

```
  shipped predictor                                        41.19
  shipped + CIRCE anchor                                   42.19
  ORACLE similarity -> softmax                             51.22
  ORACLE non-negative (NNLS) coefficients                  58.86
  ORACLE signed coefficients on the 150-row basis          65.03
  ORACLE magnitudes, our signs                             67.91
  PERFECT                                                  88.64
```

**Every oracle rung is fitted to the measured truth and therefore absorbs some of its
noise.** The distortion is worst for per-entry oracles and mildest for span-limited ones.
Noise-corrected, the best a real model could do:

```
  fixing SIGNS alone         ~46
  fixing MAGNITUDES alone    ~53
```

Closed, with numbers, so they are not retried: post-hoc transforms; spectral reshaping;
per-gene calibration and per-gene temperature; the DES nomination (a GBM on 2.7M labelled
pairs loses to `fr + 0.2|SPEC|`); similarity via PCA denoising, multi-donor pooling, graph
diffusion, CORUM, co-expression, CCA alignment, supervised ridge, GBM and a listwise
ranking loss — top-5 recall of the true nearest pool row sits at 0.25–0.33 in all of them;
signed/whitened/hubness-corrected combiners; a genome-scale matched-cell-state donor, which
reproduces every K562 failure; and a steady-state PINN, which fits in-sample at cosine
0.376 and generalises at 0.019.

The remaining lever with a measured positive return is **H1 pool size**, at ~2.1 points per
doubling, with no selection strategy beating random at the margin.